## GEOJSON Generator - 1 km^2 Bounding Box

In [ ]:
# import packages
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import os
import json
import requests
from rasterio.mask import mask
from pyproj import Proj, transform
from shapely.geometry import mapping
from shapely.geometry import shape
import rasterio as rio
import stackstac
import pystac_client
import xarray as xr
import planetary_computer

### Load data of SNOTEL stations

In [ ]:
# Load dataframe - this must have lat/lon columns! :
df = pd.read_csv('../import/df/CO_testsubset.csv') # 5 CO site example df

In [ ]:
# Specify the columns for latitude and longitude that match dataframe:
lat_col = 'latitude_exact'
lon_col = 'longitude_exact'

# Define output folder for GEOJSONs
geojson_output_folder = '../import/geojsons'

### Loop through dataframe and generate/save GEOJSONs

In [ ]:
# Ensure output folder exists
os.makedirs(geojson_output_folder, exist_ok=True)

# Loop through each row in the df to generate and save individual GeoJSON bouding boxes
for index, row in df.iterrows():
    # Get the center point
    lat = row[lat_col]
    lon = row[lon_col]

    # Define the bounds of the 1 km^2 box (0.5 degrees in lat/lon)
    half_side = 0.005  # approx 500 m 
    minx = lon - half_side
    maxx = lon + half_side
    miny = lat - half_side
    maxy = lat + half_side

    # create the box
    polygon = box(minx, miny, maxx, maxy)

    # Create a GeoJSON feature with CRS included in properties
    geojson_feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [list(polygon.exterior.coords)],
        },
        "properties": {
            "site_name": row['site_name'],
            "state": row['state'],
            "site": row['site'],
            "crs": "urn:ogc:def:crs:EPSG::4326"  # WGS84 Lat/Lon
        }
    }

    # Save the bounding box/GeoJSON using state and site number naming convention
    output_file = os.path.join(geojson_output_folder, f"{row['state']}_{str(row['site']).replace(' ', '_')}.geojson")
    with open(output_file, 'w') as f:
        json.dump(geojson_feature, f, indent=2)

    print(f"GeoJSON saved to {output_file}")